## The 2D Ising Model Near Criticality

In this project, we investigate the two–dimensional Ising model near its critical temperature. 
The main goal is to compare the performance of two Monte Carlo algorithms:

- The local Metropolis algorithm
- The cluster-based Swendsen–Wang algorithm

We focus on their behavior near the critical temperature $T_c$, where 
critical slowing down becomes significant.

In particular, we measure:

- The energy density $E/N$
- The magnetization $m$
- The autocorrelation function $\rho(\delta)$
- The integrated autocorrelation time $\tau_{\mathrm{int}}$

The reduction of $\tau_{\mathrm{int}}$ near $T_c$ serves as the main quantitative comparison between the two algorithms.

## The 2D Ising Model

The Hamiltonian of the two–dimensional Ising model is given by

$$
H(\sigma) = -J \sum_{\langle i j \rangle} \sigma_i \sigma_j - h \sum_i \sigma_i ,
$$

where:

- $\sigma_i \in \{-1, +1\}$ are classical spin variables,
- $J$ is the coupling constant,
- $h$ is an external magnetic field,
- $\langle i j \rangle$ denotes nearest-neighbor pairs.

The Boltzmann weight of a configuration is

$$
P(\sigma) \propto e^{-\beta H(\sigma)}, 
\quad
\beta = \frac{1}{k_B T}.
$$

Throughout this project, we set $k_B = 1$ and $J = 1$ unless stated otherwise.


## Lattice geometry and lookup tables

We consider a square lattice of size $L_x\times L_y$ with periodic boundary conditions.  
Sites are labeled by a single index
$$
n(x,y)=xL_y+y,\qquad N=L_xL_y,
$$
so the spin configuration (the Monte Carlo state) is stored as a 1D array $\sigma\in\{-1,+1\}^N$.

To avoid recomputing neighbor indices during the simulation, we precompute two *static* geometry tables:

- **Neighbor table** `nbrs` of shape $(N,4)$: for each site $n$, `nbrs[n] = (right, left, down, up)`.  
  This is the “hopping table” used for fast local $\Delta E$ in Metropolis.

- **Bond list** `bonds` of shape $(2N,2)$: an edge list of nearest-neighbor pairs $(i,j)$, storing each undirected bond once (right + down).  

  This is convenient for sums over $\langle ij\rangle$ and for Swendsen–Wang bond activation on edges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
def build_bonds(Lx, Ly):
    N = Lx * Ly
    bonds = np.empty((2*N, 2), dtype=np.int32)
    k = 0
    for x in range(Lx):
        for y in range(Ly):
            n = x*Ly + y 
            bonds[k] = (n, x * Ly + ((y + 1) % Ly)); k += 1
            bonds[k] = (n, ((x + 1) % Lx) * Ly + y); k += 1
    return bonds

def build_neighbours(Lx: int, Ly: int) -> np.ndarray:
    N = Lx * Ly
    nbrs = np.empty((N, 4), dtype=np.int32)
    for x in range(Lx):
        for y in range(Ly):
            n = x * Ly + y
            nbrs[n, 0] = x * Ly + ((y + 1) % Ly)  # right
            nbrs[n, 1] = x * Ly + ((y - 1) % Ly)  # left
            nbrs[n, 2] = ((x + 1) % Lx) * Ly + y  # down
            nbrs[n, 3] = ((x - 1) % Lx) * Ly + y  # up
    return nbrs

## Observables

For a given spin configuration $\sigma\in\{-1,+1\}^N$ we measure energy and magnetization.

### Energy

We use the Ising Hamiltonian
$$
H(\sigma) = -J \sum_{\langle ij\rangle}\sigma_i\sigma_j - h \sum_i \sigma_i,
$$
where $\langle ij\rangle$ denotes nearest neighbors and periodic boundary conditions are assumed.

In the code, the nearest-neighbor sum is implemented via the precomputed edge list `bonds`
(which stores each undirected bond exactly once), so that
$$
\sum_{\langle ij\rangle}\sigma_i\sigma_j \;\longleftrightarrow\; \sum_{(i,j)\in\texttt{bonds}}\sigma_i\sigma_j.
$$

We also use the energy density (energy per spin):
$$
e = \frac{H}{N}.
$$

### Magnetization

The total magnetization and magnetization density are
$$
M = \sum_i \sigma_i,
\qquad
m = \frac{1}{N}\sum_i \sigma_i = \frac{M}{N}.
$$

In order to locate the phase transition and quantify fluctuations, we later compute
susceptibility and heat capacity from time series:

$$
\chi = \beta\left(\langle M^2\rangle - \langle M\rangle^2\right)
\quad\text{(equivalently } \chi=\beta N(\langle m^2\rangle-\langle m\rangle^2)\text{)},
$$

$$
C_V = \beta^2\left(\langle H^2\rangle - \langle H\rangle^2\right).
$$

In [ ]:
def measure(spins, bonds, J=1.0, h=0.0):
    i = bonds[:, 0]
    j = bonds[:, 1]

    M = np.sum(spins)  # total magnetization
    E = -J * np.sum(spins[i] * spins[j]) - h * M

    N = spins.size
    m = M / N

    return {
        "E": float(E),
        "E2": float(E*E),
        "E_density": float(E / N),
        "M": float(M),
        "M2": float(M*M),
        "m": float(m),
        "abs_m": float(abs(m)),
        "m2": float(m*m),
    }

## Metropolis Monte Carlo (local updates)

We construct a Markov chain in configuration space by proposing local spin flips $\sigma_i \to -\sigma_i$.

For a proposed flip at site $i$, the energy change is

$$
\Delta E
= H(\sigma') - H(\sigma)
= 2\sigma_i \left(J \sum_{j\in nn(i)} \sigma_j + h\right),
$$

where $nn(i)$ are the four nearest neighbors of site $i$.

The Metropolis acceptance rule is

$$
P_{\text{acc}} = \min\left(1, e^{-\beta \Delta E}\right).
$$

In [ ]:
def metropolis_sweep(spins, nbrs, beta, rng, J=1.0, h=0.0):
    N = spins.size
    acc = 0

    for _ in range(N):
        n = int(rng.integers(0, N))
        s = spins[n]

        nn_sum = spins[nbrs[n]].sum()
        dE = 2.0 * s * (J * nn_sum + h)

        if dE <= 0.0 or rng.random() < np.exp(-beta * dE):
            spins[n] = -s
            acc += 1

    return acc / N


def run_metropolis(T, bonds, nbrs, n_therm=2000, n_meas=2000, init="random", seed=0, J=1.0, h=0.0, kB=1.0):
    rng = np.random.default_rng(seed)
    beta = 1.0 / (kB * T)
    N = bonds.max() + 1  # or nbrs.shape[0]

    if init == "random":
        spins = rng.choice([-1, 1], size=N).astype(np.int8)
    elif init == "ordered_plus":
        spins = np.ones(N, dtype=np.int8)
    elif init == "ordered_minus":
        spins = -np.ones(N, dtype=np.int8)
    else:
        raise ValueError("init must be 'random', 'ordered_plus', or 'ordered_minus'")

    # thermalization
    for _ in range(n_therm):
        metropolis_sweep(spins, nbrs, beta, rng, J=J, h=h)

    # measurement
    E = np.empty(n_meas)
    m = np.empty(n_meas)
    abs_m = np.empty(n_meas)
    acc = np.empty(n_meas)

    for t in range(n_meas):
        acc[t] = metropolis_sweep(spins, nbrs, beta, rng, J=J, h=h)
        obs = measure(spins, bonds, J=J, h=h)
        E[t] = obs["E_density"]
        m[t] = obs["m"]
        abs_m[t] = abs(m[t])

    return E, m, abs_m, acc

def plot_timeseries(E, m, title_prefix):
    plt.figure()
    plt.plot(E)
    plt.title(f"{title_prefix}: E/N vs sweep")
    plt.xlabel("sweep")
    plt.ylabel("E/N")
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(m)
    plt.title(f"{title_prefix}: m vs sweep")
    plt.xlabel("sweep")
    plt.ylabel("m")
    plt.grid(True)
    plt.show()

In [ ]:
# Test run Metropolis
L = 32
N = L*L
bonds = build_bonds(L, L)
nbrs = build_neighbours(L, L)

J = 1.0
h = 0.0
kB = 1.0

for T in [1.5, 3.5]:
    for init in ["random", "ordered_plus"]:
        E, m, abs_m, acc = run_metropolis(
            T, bonds, nbrs,
            n_therm=2000, n_meas=2000,
            init=init, seed=0, J=J, h=h, kB=kB
        )

        print(f"Metropolis | T={T:.2f} | init={init:11s} | <E/N>={E.mean(): .3f} | <m>={m.mean(): .3f} | <|m|>={abs_m.mean(): .3f} | <acc>={acc.mean(): .3f}")
        plot_timeseries(E, m, title_prefix=f"Metropolis (T={T}, {init})")

## Interpreting the Metropolis test runs

### Acceptance rate

The acceptance rate $(\langle a\rangle$) is small at low temperature because most proposed flips increase the energy:
flipping a spin inside an ordered domain typically breaks several aligned bonds, so $(\Delta E > 0$) and the proposal is suppressed by $(e^{-\beta\Delta E}$).

At high temperature, thermal fluctuations dominate and many moves are accepted, so $(\langle a\rangle$) is much larger.

### Magnetization

At $(h=0$) and finite lattice size, the exact equilibrium satisfies $(\langle m\rangle = 0$) by symmetry.
Below $(T_c$) the system spends long times in a state with $(m \approx +m_0$) or $(m \approx -m_0$),
so $(\langle |m|\rangle$) is the more informative quantity.

A key practical issue is **slow tunnelling** between $(+m_0$) and $(-m_0$) for local updates:
Metropolis can remain trapped for a long time, which is a manifestation of critical slowing down.

## Swendsen–Wang update (cluster algorithm)

The Swendsen–Wang algorithm replaces local spin flips by non-local cluster flips.

For each nearest-neighbor bond $(i,j)$ we activate an edge with probability

$$
p_{\text{bond}} = 1 - e^{-2\beta J}
$$

only if the spins are parallel ($\sigma_i=\sigma_j$).  
Activated bonds define a graph on lattice sites; connected components of this graph are the clusters.

Finally, each cluster is flipped as a whole with probability $1/2$.

This update satisfies detailed balance and strongly reduces critical slowing down near $T_c$.

In [ ]:
def sw_update(spins, bonds, beta, rng, J=1.0, return_details=False):
    N = spins.size
    p_bond = 1.0 - np.exp(-2.0 * beta * J)

    i = bonds[:, 0]
    j = bonds[:, 1]

    parallel = (spins[i] == spins[j])
    active = parallel & (rng.random(size=bonds.shape[0]) < p_bond)

    A = csr_matrix(
        (np.ones(active.sum(), dtype=np.int8), (i[active], j[active])),
        shape=(N, N)
    )
    A = A + A.T

    n_comp, labels = connected_components(A, directed=False)

    flip = rng.random(n_comp) < 0.5
    spins_new = spins.copy()
    spins_new[flip[labels]] *= -1

    if return_details:
        return spins_new, {
            "p_bond": p_bond,
            "active": active,
            "labels": labels,
            "n_comp": n_comp,
        }

    return spins_new

In [ ]:
# SW test runner

L = 32
N = L**2

rng = np.random.default_rng(0)   # фиксируем seed для воспроизводимости
bonds = build_bonds(L, L)

J = 1.0
h = 0.0
kB = 1.0

def run_sw_test(T, n_therm=500, n_meas=1000, init="random", seed=0, with_cluster_stats=False):
    rng = np.random.default_rng(seed)
    beta = 1.0 / T

    if init == "random":
        spins = rng.choice([-1, 1], size=N).astype(np.int8)
    elif init == "ordered":
        spins = np.ones(N, dtype=np.int8)
    else:
        raise ValueError("init must be 'random' or 'ordered'")

    # thermalization
    for _ in range(n_therm):
        spins = sw_update(spins, bonds, beta, rng, J=J)

    # measurement arrays
    E = np.empty(n_meas)
    m = np.empty(n_meas)
    abs_m = np.empty(n_meas)

    if with_cluster_stats:
        n_clusters = np.empty(n_meas, dtype=int)
        max_cluster = np.empty(n_meas, dtype=int)

    for t in range(n_meas):
        if with_cluster_stats:
            spins, info = sw_update(spins, bonds, beta, rng, J=J, return_details=True)
            n_clusters[t] = info["n_comp"]
            sizes = np.bincount(info["labels"], minlength=info["n_comp"])
            max_cluster[t] = sizes.max()
        else:
            spins = sw_update(spins, bonds, beta, rng, J=J)

        obs = measure(spins, bonds, J=J, h=h)
        E[t] = obs["E_density"]
        m[t] = obs["m"]
        abs_m[t] = abs(m[t])

    if with_cluster_stats:
        return E, m, abs_m, n_clusters, max_cluster
    return E, m, abs_m


def plot_series(E, m, title_prefix):
    plt.figure()
    plt.plot(E)
    plt.title(f"{title_prefix}: E/N vs step")
    plt.xlabel("Measurement step")
    plt.ylabel("E/N")
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(m)
    plt.title(f"{title_prefix}: m vs step")
    plt.xlabel("Measurement step")
    plt.ylabel("m")
    plt.grid(True)
    plt.show()


for T in [1.5, 3.5]:
    init ="random"
    out = run_sw_test(T, n_therm=200, n_meas=500, init=init, seed=0, with_cluster_stats=True)
    E, m, abs_m, n_clusters, max_cluster = out

    print(f"SW  | T={T:4.2f} | init={init:7s} | <E/N>={E.mean(): .3f} | <m>={m.mean(): .3f} | <|m|>={abs_m.mean(): .3f}")
    print(f"     clusters: <n>={n_clusters.mean():.1f},  <max size>={max_cluster.mean():.1f} (out of N={N})")

    plot_series(E, m, title_prefix=f"SW (T={T}, {init} init)")

    # cluster diagnostics plots
    plt.figure()
    plt.plot(n_clusters)
    plt.title(f"SW: number of clusters vs step (T={T}, {init} init)")
    plt.xlabel("Measurement step")
    plt.ylabel("n_clusters")
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(max_cluster)
    plt.title(f"SW: largest cluster size vs step (T={T}, {init} init)")
    plt.xlabel("Measurement step")
    plt.ylabel("max_cluster_size")
    plt.grid(True)
    plt.show()


## Cluster statistics: what do we expect?

In the Swendsen–Wang update we activate a bond only between **parallel nearest neighbours** with probability

$$
p_{\text{bond}} = 1 - e^{-2\beta J}.
$$

The activated bonds define a graph on lattice sites; its **connected components** are the clusters.
Clusters therefore represent **correlated spin domains** at the current temperature.

### Low temperature $(T < T_c$)

Spins are mostly aligned and correlations are long-ranged.  
Since $(p_{\text{bond}}$) is also large, the activated-bond graph typically contains a **system-spanning (percolating) cluster**.

Typical diagnostics:
- small number of clusters: $(\langle n_{\text{clusters}}\rangle \ll N$)
- largest cluster size close to the full lattice: $(s_{\max} \approx N$)

Even if a giant cluster exists, each cluster is flipped with probability $(1/2$) (at $(h=0$)),
so the total magnetization can frequently switch sign.  
This is why $(\langle m\rangle \approx 0$) but $(\langle |m| \rangle$) is close to 1 below $(T_c$).

### High temperature $(T > T_c$)

Spins are disordered and the correlation length is short.
Parallel neighbours are less common, and even then bonds are activated only with probability $(p_{\text{bond}}$).

Typical diagnostics:
- large number of clusters (many small components)
- largest cluster is small compared to $(N$): $(s_{\max} \ll N$)